In [ ]:
"""
WESAD ECG Analysis: Complete Pipeline
=====================================
This notebook performs three main analyses:
1. Statistical comparison between 5-min and 30-sec ECG chunks
2. ML model comparison between 30-sec and 5-min features
3. FFT analysis for stress vs non-stress comparison
"""

# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our custom classes
import sys
sys.path.append('../src')

from data import Data
from features import Features
from visualization import Visualization
from correlation import Correlation
from ml import ML

# Machine learning models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

# Set random seed for reproducibility
np.random.seed(42)

# Initialize all classes
data_processor = Data(fs=700)
feature_extractor = Features(fs=700)
viz = Visualization()
corr = Correlation()
ml_evaluator = ML(random_state=42)

print("✅ All classes initialized successfully!")
print(f"Sampling frequency: {data_processor.fs} Hz")

# ============================================
# PART 1: LOAD AND PREPARE DATA
# ============================================

# Path to your WESAD data
DATA_PATH = "../data/raw/WESAD/"  # Update this path

# Load dataset
print("\n📊 Loading WESAD dataset...")
ecgs_list, labels_list = data_processor.read_dataset(DATA_PATH)
print(f"Loaded {len(ecgs_list)} subjects")
print(f"ECG lengths: {[len(ecg) for ecg in ecgs_list]}")
print(f"Label shapes: {[len(label) for label in labels_list]}")

# Filter for stress (label=2) and baseline (label=1) only
# Label mapping from WESAD:
# 1 = baseline (non-stress)
# 2 = stress
# 3 = amusement (ignore for now)
# 4 = meditation (ignore for now)

print("\n🔍 Filtering for baseline (non-stress) and stress conditions...")
filtered_ecgs = []
filtered_labels = []

for ecg, labels in zip(ecgs_list, labels_list):
    # Create mask for baseline AND stress (exclude amusement/meditation)
    mask = (labels == 1) | (labels == 2)
    filtered_ecgs.append(ecg[mask])
    filtered_labels.append(labels[mask])

# Now get separate lists for baseline and stress
baseline_ecgs = []
stress_ecgs = []

for ecg, labels in zip(filtered_ecgs, filtered_labels):
    baseline_ecgs.append(ecg[labels == 1])
    stress_ecgs.append(ecg[labels == 2])

print(f"Subjects with baseline data: {len(baseline_ecgs)}")
print(f"Subjects with stress data: {len(stress_ecgs)}")

# ============================================
# PART 2: EXTRACT FEATURES FOR DIFFERENT CHUNK SIZES
# ============================================

print("\n" + "="*60)
print("PART 2: FEATURE EXTRACTION")
print("="*60)

# Define chunk sizes
CHUNK_SIZES = {
    '30sec': 30,      # 30 seconds
    '5min': 300       # 5 minutes
}

# Store features for different conditions
features_dict = {
    '30sec_baseline': [],
    '30sec_stress': [],
    '5min_baseline': [],
    '5min_stress': []
}

print("\n🔄 Extracting features from 30-second chunks...")

# Process 30-second chunks
for size_name, duration in CHUNK_SIZES.items():
    print(f"\nProcessing {size_name} chunks ({duration} seconds)...")
    
    for condition_name, ecg_list in [('baseline', baseline_ecgs), ('stress', stress_ecgs)]:
        print(f"  Condition: {condition_name}")
        
        all_features = []
        
        for subject_idx, subject_ecg in enumerate(ecg_list):
            # Get sequential chunks
            chunks = data_processor.get_chunk_ecg(subject_ecg, time_in_sec=duration)
            
            # Extract features from each chunk
            for chunk in chunks:
                features = feature_extractor.get_hrv_features(chunk)
                all_features.append(features)
        
        # Store features
        key = f"{size_name}_{condition_name}"
        features_dict[key] = all_features
        print(f"    Extracted {len(all_features)} feature vectors")

# Create DataFrames for each condition
dfs = {}
for key, features_list in features_dict.items():
    df = feature_extractor.get_dataframe(features_list)
    # Add condition label
    condition = 'stress' if 'stress' in key else 'baseline'
    df['condition'] = condition
    df['chunk_size'] = '30sec' if '30sec' in key else '5min'
    dfs[key] = df
    print(f"\n{key} DataFrame shape: {df.shape}")

# ============================================
# PART 3: STATISTICAL COMPARISON (5-min vs 30-sec)
# ============================================

print("\n" + "="*60)
print("PART 3: STATISTICAL COMPARISON (5-min vs 30-sec)")
print("="*60)

# Compare 5-min vs 30-sec features for baseline condition
print("\n📈 Comparing 5-min vs 30-sec chunks (Baseline condition)...")

feature_names = ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']

# Create comparison DataFrames
comparison_results = []

for feature in feature_names:
    # Get feature values for 30-sec and 5-min baseline
    f_30sec = dfs['30sec_baseline'][feature].dropna().values
    f_5min = dfs['5min_baseline'][feature].dropna().values
    
    # Calculate statistics
    r, p_value = corr.get_r(f_30sec, f_5min)
    icc, icc_ci = corr.get_icc(f_30sec, f_5min)
    mae = corr.get_mae(f_30sec, f_5min)
    
    comparison_results.append({
        'feature': feature,
        'pearson_r': r,
        'p_value': p_value,
        'icc': icc,
        'icc_ci_lower': icc_ci[0],
        'icc_ci_upper': icc_ci[1],
        'mae': mae,
        'interpretation': corr.interpret_correlation(r)
    })

comparison_df = pd.DataFrame(comparison_results)
print("\n📊 Statistical comparison results (5-min vs 30-sec baseline):")
display(comparison_df.round(4))

# Visualize correlation between 5-min and 30-sec features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(feature_names):
    ax = axes[idx]
    f_30sec = dfs['30sec_baseline'][feature].dropna().values
    f_5min = dfs['5min_baseline'][feature].dropna().values
    
    ax.scatter(f_30sec, f_5min, alpha=0.5, color='#2d7a3e')
    ax.plot([f_30sec.min(), f_30sec.max()], [f_30sec.min(), f_30sec.max()], 
            'r--', alpha=0.5, label='Identity line')
    ax.set_xlabel(f'{feature}\n(30-sec chunks)')
    ax.set_ylabel(f'{feature}\n(5-min chunks)')
    
    r, _ = corr.get_r(f_30sec, f_5min)
    ax.set_title(f'{feature}\nR = {r:.3f}')
    ax.legend()

plt.suptitle('Correlation: 5-min vs 30-sec Features (Baseline Condition)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================
# PART 4: MACHINE LEARNING COMPARISON
# ============================================

print("\n" + "="*60)
print("PART 4: MACHINE LEARNING COMPARISON")
print("="*60)

# Prepare data for ML (combine baseline and stress with labels)
ml_data = {}

for chunk_size in ['30sec', '5min']:
    # Combine baseline and stress for this chunk size
    baseline_df = dfs[f'{chunk_size}_baseline'].copy()
    stress_df = dfs[f'{chunk_size}_stress'].copy()
    
    baseline_df['label'] = 0  # Non-stress
    stress_df['label'] = 1     # Stress
    
    combined_df = pd.concat([baseline_df, stress_df], ignore_index=True)
    
    # Add subject IDs for LOSO (simulate since we don't have real subject IDs)
    # In real scenario, you'd have subject_id column
    n_baseline = len(baseline_df)
    n_stress = len(stress_df)
    combined_df['subject_id'] = list(range(n_baseline)) + list(range(n_stress))
    
    ml_data[chunk_size] = combined_df
    print(f"\n{chunk_size} data shape: {combined_df.shape}")
    print(f"  Class distribution: Non-stress={n_baseline}, Stress={n_stress}")

# Define models to compare
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

# Feature columns (excluding non-feature columns)
feature_cols = feature_names

# Store results
ml_results = {}

print("\n🔄 Running LOSO Cross-Validation...")
print("-" * 50)

for chunk_size in ['30sec', '5min']:
    print(f"\n📊 Evaluating models on {chunk_size} chunks...")
    
    df = ml_data[chunk_size]
    results = ml_evaluator.eval_all(df, models, feature_cols, cv_method='loso')
    ml_results[chunk_size] = results
    
    # Display results
    for model_name, model_results in results.items():
        overall = model_results['overall']
        print(f"\n  {model_name}:")
        print(f"    Accuracy: {overall['accuracy']:.4f}")
        print(f"    F1-Score: {overall['f1']:.4f}")
        print(f"    Precision: {overall['precision']:.4f}")
        print(f"    Recall: {overall['recall']:.4f}")

# Compare models across chunk sizes
print("\n" + "="*50)
print("📊 MODEL COMPARISON: 30-sec vs 5-min chunks")
print("="*50)

comparison_data = []
for chunk_size in ['30sec', '5min']:
    for model_name, results in ml_results[chunk_size].items():
        comparison_data.append({
            'chunk_size': chunk_size,
            'model': model_name,
            'accuracy': results['overall']['accuracy'],
            'f1': results['overall']['f1'],
            'precision': results['overall']['precision'],
            'recall': results['overall']['recall']
        })

comparison_ml_df = pd.DataFrame(comparison_data)
display(comparison_ml_df.round(4))

# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['accuracy', 'f1', 'precision', 'recall']
x = np.arange(len(metrics))
width = 0.35

for idx, chunk_size in enumerate(['30sec', '5min']):
    ax = axes[idx]
    chunk_data = comparison_ml_df[comparison_ml_df['chunk_size'] == chunk_size]
    
    for i, model in enumerate(models.keys()):
        model_data = chunk_data[chunk_data['model'] == model]
        values = [model_data[m].values[0] for m in metrics]
        ax.bar(x + i*width, values, width, label=model)
    
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score')
    ax.set_title(f'{chunk_size} chunks performance')
    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim([0, 1])
    ax.grid(True, alpha=0.3)

plt.suptitle('Model Performance Comparison: 30-sec vs 5-min Chunks', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ============================================
# PART 5: FFT ANALYSIS (Stress vs Non-stress)
# ============================================

print("\n" + "="*60)
print("PART 5: FFT ANALYSIS - Stress vs Non-stress")
print("="*60)

def compute_fft_features(ecg_signal, fs=700):
    """
    Compute FFT-based frequency features from ECG signal.
    """
    # Remove DC component
    ecg_signal = ecg_signal - np.mean(ecg_signal)
    
    # Compute FFT
    n = len(ecg_signal)
    freqs = np.fft.rfftfreq(n, 1/fs)
    fft_vals = np.fft.rfft(ecg_signal)
    power_spectrum = np.abs(fft_vals)**2
    
    # Define frequency bands
    # ECG frequency bands (mostly below 50 Hz)
    bands = {
        'VLF (0-0.04 Hz)': (0, 0.04),
        'LF (0.04-0.15 Hz)': (0.04, 0.15),
        'HF (0.15-0.4 Hz)': (0.15, 0.4),
        'VHF (0.4-10 Hz)': (0.4, 10),
        'All (0-50 Hz)': (0, 50)
    }
    
    features = {}
    for band_name, (low, high) in bands.items():
        mask = (freqs >= low) & (freqs <= high)
        power = np.sum(power_spectrum[mask])
        features[f'power_{band_name.replace(" ", "_").replace("(", "").replace(")", "")}'] = power
        features[f'power_log_{band_name.replace(" ", "_").replace("(", "").replace(")", "")}'] = np.log1p(power)
    
    # Add peak frequency
    peak_idx = np.argmax(power_spectrum)
    features['peak_frequency'] = freqs[peak_idx]
    features['peak_power'] = power_spectrum[peak_idx]
    
    return features

print("\n🔍 Computing FFT features for stress and non-stress conditions...")

# Sample ECG chunks for FFT analysis
fft_features = {'non_stress': [], 'stress': []}

for condition, ecg_list in [('non_stress', baseline_ecgs), ('stress', stress_ecgs)]:
    print(f"Processing {condition} condition...")
    
    for subject_idx, subject_ecg in enumerate(ecg_list):
        # Take first 30-second chunk from each subject
        chunk = data_processor.get_chunk_ecg(subject_ecg, time_in_sec=30)
        if len(chunk) > 0:
            features = compute_fft_features(chunk[0], fs=700)
            features['subject'] = subject_idx
            fft_features[condition].append(features)

# Create DataFrames
fft_non_stress_df = pd.DataFrame(fft_features['non_stress'])
fft_stress_df = pd.DataFrame(fft_features['stress'])

print(f"\nNon-stress samples: {len(fft_non_stress_df)}")
print(f"Stress samples: {len(fft_stress_df)}")

# Visualize FFT power spectrum comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot average power spectrum for each condition
fs = 700
sample_ecg_non_stress = None
sample_ecg_stress = None

# Get sample ECG signals
for ecg_list in baseline_ecgs:
    if len(ecg_list) > 0:
        sample_ecg_non_stress = data_processor.get_chunk_ecg(ecg_list[0], time_in_sec=10)[0]
        break

for ecg_list in stress_ecgs:
    if len(ecg_list) > 0:
        sample_ecg_stress = data_processor.get_chunk_ecg(ecg_list[0], time_in_sec=10)[0]
        break

if sample_ecg_non_stress is not None and sample_ecg_stress is not None:
    # Plot time domain signals
    ax1 = axes[0, 0]
    time = np.arange(len(sample_ecg_non_stress)) / fs
    ax1.plot(time, sample_ecg_non_stress, label='Non-stress', alpha=0.7, color='#2d7a3e')
    ax1.plot(time, sample_ecg_stress, label='Stress', alpha=0.7, color='#c41e3a')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude (mV)')
    ax1.set_title('ECG Signals Comparison (10 seconds)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot FFT magnitude spectra
    ax2 = axes[0, 1]
    fft_non_stress = np.abs(np.fft.rfft(sample_ecg_non_stress))
    fft_stress = np.abs(np.fft.rfft(sample_ecg_stress))
    freqs = np.fft.rfftfreq(len(sample_ecg_non_stress), 1/fs)
    
    ax2.plot(freqs, fft_non_stress, label='Non-stress', alpha=0.7, color='#2d7a3e')
    ax2.plot(freqs, fft_stress, label='Stress', alpha=0.7, color='#c41e3a')
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Magnitude')
    ax2.set_title('FFT Magnitude Spectrum')
    ax2.set_xlim([0, 20])  # Focus on relevant frequencies
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# Plot power in different frequency bands
ax3 = axes[1, 0]
power_cols = [col for col in fft_non_stress_df.columns if col.startswith('power_') and 'log' not in col]
power_non_stress = fft_non_stress_df[power_cols].mean()
power_stress = fft_stress_df[power_cols].mean()

x = np.arange(len(power_cols))
width = 0.35
ax3.bar(x - width/2, power_non_stress, width, label='Non-stress', color='#2d7a3e', alpha=0.7)
ax3.bar(x + width/2, power_stress, width, label='Stress', color='#c41e3a', alpha=0.7)
ax3.set_xlabel('Frequency Bands')
ax3.set_ylabel('Power')
ax3.set_title('Power Distribution Across Frequency Bands')
ax3.set_xticks(x)
ax3.set_xticklabels([col.replace('power_', '').replace('_', ' ') for col in power_cols], rotation=45)
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot power ratios
ax4 = axes[1, 1]
lf_hf_ratio_non_stress = (fft_non_stress_df['power_LF_0_04_0_15_Hz'] / 
                           fft_non_stress_df['power_HF_0_15_0_4_Hz']).mean()
lf_hf_ratio_stress = (fft_stress_df['power_LF_0_04_0_15_Hz'] / 
                       fft_stress_df['power_HF_0_15_0_4_Hz']).mean()

ax4.bar(['Non-stress', 'Stress'], [lf_hf_ratio_non_stress, lf_hf_ratio_stress], 
        color=['#2d7a3e', '#c41e3a'], alpha=0.7)
ax4.set_ylabel('LF/HF Ratio')
ax4.set_title('LF/HF Ratio: Non-stress vs Stress')
ax4.grid(True, alpha=0.3)

plt.suptitle('FFT Analysis: Stress vs Non-stress ECG Signals', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical comparison of FFT features
print("\n📊 Statistical comparison of FFT features:")
fft_comparison = []

for col in power_cols:
    non_stress_vals = fft_non_stress_df[col].dropna().values
    stress_vals = fft_stress_df[col].dropna().values
    
    from scipy.stats import ttest_ind
    t_stat, p_value = ttest_ind(non_stress_vals, stress_vals)
    
    # Calculate fold change
    fold_change = stress_vals.mean() / non_stress_vals.mean() if non_stress_vals.mean() > 0 else np.nan
    
    fft_comparison.append({
        'feature': col.replace('power_', ''),
        'non_stress_mean': non_stress_vals.mean(),
        'stress_mean': stress_vals.mean(),
        'fold_change': fold_change,
        'p_value': p_value,
        'significant': p_value < 0.05
    })

fft_comparison_df = pd.DataFrame(fft_comparison)
display(fft_comparison_df.round(4))

# ============================================
# PART 6: SAVE RESULTS
# ============================================

print("\n" + "="*60)
print("PART 6: SAVING RESULTS")
print("="*60)

# Save DataFrames to CSV
output_dir = Path("../results/metrics")
output_dir.mkdir(parents=True, exist_ok=True)

# Save feature DataFrames
for key, df in dfs.items():
    df.to_csv(output_dir / f"features_{key}.csv", index=False)

# Save ML comparison results
comparison_ml_df.to_csv(output_dir / "ml_comparison_results.csv", index=False)

# Save FFT comparison results
fft_comparison_df.to_csv(output_dir / "fft_comparison_results.csv", index=False)

# Save statistical comparison
comparison_df.to_csv(output_dir / "statistical_comparison_5min_vs_30sec.csv", index=False)

print(f"\n✅ Results saved to {output_dir}")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*60)
print("ANALYSIS SUMMARY")
print("="*60)

print("\n1️⃣ STATISTICAL COMPARISON (5-min vs 30-sec):")
print(f"   - Best correlated feature: {comparison_df.loc[comparison_df['pearson_r'].idxmax(), 'feature']}")
print(f"     (R = {comparison_df['pearson_r'].max():.3f})")
print(f"   - Worst correlated feature: {comparison_df.loc[comparison_df['pearson_r'].idxmin(), 'feature']}")
print(f"     (R = {comparison_df['pearson_r'].min():.3f})")

print("\n2️⃣ MACHINE LEARNING COMPARISON:")
for chunk_size in ['30sec', '5min']:
    best_model = comparison_ml_df[comparison_ml_df['chunk_size'] == chunk_size].iloc[
        comparison_ml_df[comparison_ml_df['chunk_size'] == chunk_size]['accuracy'].idxmax()
    ]
    print(f"   {chunk_size} chunks - Best model: {best_model['model']} (Accuracy: {best_model['accuracy']:.4f})")

print("\n3️⃣ FFT ANALYSIS (Stress vs Non-stress):")
significant_features = fft_comparison_df[fft_comparison_df['significant'] == True]
print(f"   Significant FFT features: {len(significant_features)} out of {len(fft_comparison_df)}")
for _, row in significant_features.iterrows():
    print(f"   - {row['feature']}: {row['fold_change']:.2f}x change (p={row['p_value']:.4f})")

print("\n✅ Analysis complete!")